# DSCI 550 — Assignment 4: Neural Networks and Deep Learning Fundamentals

**100 points · PyTorch · Dense feedforward networks**

This assignment focuses on the core ideas needed to understand and train deep neural networks:

- dense feedforward networks;
- ReLU activations;
- classification loss;
- backpropagation and gradients;
- gradient-based parameter updates;
- mini-batch training and evaluation;
- dropout and early stopping.

The scope follows the introductory, dense-network, backpropagation/gradient-descent, and regularization material in **Chapter 13, _Neural Nets and Deep Learning_, Mining of Massive Datasets**. CNNs, RNNs, and LSTMs are intentionally **not** part of this assignment.

## Dataset

We use scikit-learn's **Forest Cover Type (`fetch_covtype`)** dataset, a real tabular multiclass classification dataset. The task is to predict one of seven forest cover types from cartographic attributes.

The provided setup code downloads/caches the dataset and then creates the dataset used for this assignment in a fully reproducible way:

- **54 numerical input features**;
- **7 target classes**;
- exactly **2,000 examples per class** are selected with random seed 42;
- **14,000 examples total**;
- fixed **70% / 15% / 15%** train-validation-test split;
- labels are mapped from 1–7 to **0–6** for PyTorch cross-entropy;
- features are standardized using statistics from the training split only.

Because the sampling and splitting code is provided and uses fixed seeds, everyone works with the same subset and splits. You do **not** need to clean, sample, standardize, or split the data yourself.

> The first execution of the setup cell may take longer because scikit-learn may need to download the dataset. Later runs normally use its local cache.

## Required libraries

| Task | Required libraries|
|---|---|
| Tensor operations | PyTorch |
| Dense layers | `torch.nn.Linear` |
| Activation | `torch.nn.ReLU` / `torch.relu` |
| Classification loss | PyTorch cross-entropy |
| Baseline MLP | 54→128 → ReLU → 128→64 → ReLU → 64→7 logits |
| Regularized MLP | Same MLP with dropout after each hidden ReLU |
| Optimization in hidden tests | SGD with a supplied learning rate |
| Random seed | 42 unless the question supplies another seed |

## Points

| Part | Topic | Points |
|---|---|---:|
| 1 | Forward propagation & MLPs | 20 |
| 2 | Loss, backpropagation & optimization | 25 |
| 3 | Mini-batch training & evaluation | 20 |
| 4 | Regularization | 15 |
| 5 | End-to-end training | 20 |
| **Total** |  | **100** |



In [25]:
# SETUP — run this cell first. Do not modify.

import copy
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

RANDOM_STATE = 42
INPUT_DIM = 54
NUM_CLASSES = 7
SAMPLES_PER_CLASS = 2000

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(1)

# -----------------------------------------------------------------------------
# Forest Cover Type dataset (sklearn.datasets.fetch_covtype)
# -----------------------------------------------------------------------------
# The first run downloads the dataset through scikit-learn and caches it.
# All students use the same deterministic balanced subset below.
_covtype = fetch_covtype(as_frame=False)
_X_all = _covtype.data.astype(np.float32)
_y_all = _covtype.target.astype(np.int64) - 1   # map labels 1..7 to 0..6

# Select exactly 2,000 examples from each of the seven classes using a fixed
# random seed. This produces the same 14,000-example subset for everyone.
_rng = np.random.RandomState(RANDOM_STATE)
_selected_indices = []
for _class_id in range(NUM_CLASSES):
    _class_indices = np.flatnonzero(_y_all == _class_id)
    _selected_indices.append(
        _rng.choice(_class_indices, size=SAMPLES_PER_CLASS, replace=False)
    )
_selected_indices = np.concatenate(_selected_indices)
_rng.shuffle(_selected_indices)

_X = _X_all[_selected_indices]
_y = _y_all[_selected_indices]

# Fixed 70% / 15% / 15% train-validation-test split.
X_TRAIN_RAW, _X_temp, Y_TRAIN_RAW, _y_temp = train_test_split(
    _X,
    _y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=_y,
)
X_VAL_RAW, X_TEST_RAW, Y_VAL_RAW, Y_TEST_RAW = train_test_split(
    _X_temp,
    _y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=_y_temp,
)

# Standardize with statistics learned from the training split only.
_SCALER = StandardScaler()
X_TRAIN = _SCALER.fit_transform(X_TRAIN_RAW).astype(np.float32)
X_VAL = _SCALER.transform(X_VAL_RAW).astype(np.float32)
X_TEST = _SCALER.transform(X_TEST_RAW).astype(np.float32)

Y_TRAIN = Y_TRAIN_RAW.astype(np.int64)
Y_VAL = Y_VAL_RAW.astype(np.int64)
Y_TEST = Y_TEST_RAW.astype(np.int64)

# Convert to PyTorch tensors.
X_TRAIN = torch.tensor(X_TRAIN, dtype=torch.float32)
Y_TRAIN = torch.tensor(Y_TRAIN, dtype=torch.long)
X_VAL = torch.tensor(X_VAL, dtype=torch.float32)
Y_VAL = torch.tensor(Y_VAL, dtype=torch.long)
X_TEST = torch.tensor(X_TEST, dtype=torch.float32)
Y_TEST = torch.tensor(Y_TEST, dtype=torch.long)

TRAIN_DATASET = TensorDataset(X_TRAIN, Y_TRAIN)
VAL_DATASET = TensorDataset(X_VAL, Y_VAL)
TEST_DATASET = TensorDataset(X_TEST, Y_TEST)

_train_generator = torch.Generator().manual_seed(RANDOM_STATE)
TRAIN_LOADER = DataLoader(
    TRAIN_DATASET,
    batch_size=64,
    shuffle=True,
    generator=_train_generator,
    num_workers=0,
)
VAL_LOADER = DataLoader(
    VAL_DATASET, batch_size=128, shuffle=False, num_workers=0
)
TEST_LOADER = DataLoader(
    TEST_DATASET, batch_size=128, shuffle=False, num_workers=0
)

print('Dataset: sklearn Forest Cover Type (balanced deterministic subset)')
print('Train:', tuple(X_TRAIN.shape), tuple(Y_TRAIN.shape))
print('Validation:', tuple(X_VAL.shape), tuple(Y_VAL.shape))
print('Test:', tuple(X_TEST.shape), tuple(Y_TEST.shape))

Dataset: sklearn Forest Cover Type (balanced deterministic subset)
Train: (9800, 54) (9800,)
Validation: (2100, 54) (2100,)
Test: (2100, 54) (2100,)


## Part 1 — Forward Propagation & MLPs (20 points)

### Q1 — Dense layer followed by ReLU (6 pts)

Implement `linear_relu_forward(X, W, b)`.

- `X` has shape `(batch_size, input_features)`.
- `W` has shape `(output_features, input_features)`.
- `b` has shape `(output_features,)`.
- Apply the dense linear transformation using `W` and `b`, then apply ReLU element-wise.
- Return the resulting tensor.

Use PyTorch tensor operations. Do not create an `nn.Linear` module inside this function.

In [26]:
def linear_relu_forward(X, W, b):
    # YOUR CODE HERE
    product = torch.matmul(X,W.T)
    res = torch.add(product,b)
    relu_layer = nn.ReLU()

    return relu_layer(res)
    

### Q2 — Build a dense feedforward network (8 pts)

Implement `build_mlp()` using PyTorch.

Return an `nn.Sequential` model with the following architecture, in this order:

1. dense layer from **54 input features** to 128 hidden units;
2. ReLU;
3. dense layer from 128 to 64 hidden units;
4. ReLU;
5. dense output layer from 64 to **7 logits**.

Do **not** add softmax to the model. The classification loss will operate directly on logits.

In [ ]:
def build_mlp():
    # YOUR CODE HERE
    return nn.Sequential(nn.Linear(54, 128),nn.ReLU(),nn.Linear(128, 64),nn.ReLU(),nn.Linear(64, 7))
    

### Q3 — Count trainable parameters (6 pts)

Implement `count_trainable_parameters(model)`.

Return the total number of scalar parameters in `model` whose `requires_grad` flag is `True`.

Your function must work for an arbitrary PyTorch model, not only the MLP from Q2.

In [28]:
def count_trainable_parameters(model):
    # YOUR CODE HERE
    count = 0
    for par in model.parameters():

        if par.requires_grad == True:

            count += par.numel()

    return count


## Part 2 — Loss, Backpropagation & Optimization (25 points)

### Q4 — Cross-entropy classification loss (7 pts)

Implement `classification_loss(logits, targets)`.

- `logits` has shape `(batch_size, num_classes)`.
- `targets` contains integer class labels.
- Return PyTorch cross-entropy loss using its default reduction.

Return the scalar loss tensor, not a Python list or per-example loss vector.

In [29]:
def classification_loss(logits, targets):
    # YOUR CODE HERE
    CE = nn.CrossEntropyLoss()
    return CE(logits,targets)


### Q5 — Compute the first-layer gradient (8 pts)

Implement `first_layer_gradient(model, X, y)`.

The function must:

1. clear any existing parameter gradients;
2. run a forward pass on `X`;
3. compute cross-entropy loss against `y`;
4. run backpropagation;
5. locate the **first `nn.Linear` layer** in the model;
6. return a detached clone of that layer's **weight gradient**.

Do not update the model parameters in this function.

In [30]:
def first_layer_gradient(model, X, y):
    # YOUR CODE HERE
    model.zero_grad()
    pass_ = model(X)

    CE = nn.CrossEntropyLoss()
    loss = CE(pass_, y)

    loss.backward()

    for  param in model.modules():

        if isinstance(param, nn.Linear):

            return param.weight.grad.detach().clone()

    return None

### Q6 — Perform one optimizer step (10 pts)

Implement `train_one_batch(model, X, y, optimizer)`.

For exactly one mini-batch:

1. put the model in training mode;
2. clear old gradients;
3. run a forward pass;
4. compute cross-entropy loss;
5. backpropagate;
6. call the supplied optimizer once;
7. return the loss **before the parameter update** as a Python `float`.

Do not create a new optimizer inside the function.

In [31]:
def train_one_batch(model, X, y, optimizer):

    model.train()
    model.zero_grad()
    pass_ = model(X)

    CE = nn.CrossEntropyLoss()
    loss = CE(pass_, y)

    loss.backward()

    optimizer.step()

    return loss.item()

## Part 3 — Mini-Batch Training & Evaluation (20 points)

### Q7 — Train for one epoch (10 pts)

Implement `train_epoch(model, loader, optimizer)`.

Process every mini-batch in `loader` exactly once using cross-entropy loss and the supplied optimizer. Put the model in training mode.

Return the **average loss per example** over the full epoch as a Python `float`. If the final batch is smaller, it must be weighted by its actual number of examples rather than treating every batch equally.

In [32]:
def train_epoch(model, loader, optimizer):
    # YOUR CODE HERE
    model.train()
    

    total_loss = 0.0
    total_samples = 0


    for inputs, label in loader:

        optimizer.zero_grad()

        pass_ = model(inputs)
        CE = nn.CrossEntropyLoss()

        loss_ = CE(pass_, label)
        loss_.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss_.item() * batch_size
        total_samples += batch_size

    if total_samples > 0:

        return total_loss / total_samples

    return 0.0
    



### Q8 — Evaluate a model (10 pts)

Implement `evaluate(model, loader)`.

The function must:

- put the model in evaluation mode;
- disable gradient tracking during evaluation;
- process all examples in `loader`;
- compute the average cross-entropy loss per example;
- compute classification accuracy using the largest-logit class as the prediction.

Return `(average_loss, accuracy)` as two Python `float` values. Accuracy should be between 0 and 1. Do not modify model parameters.

In [33]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct_preds = 0
    total_samples = 0
    avg_loss = 0
    accuracy = 0
    
    CE = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for inputs, labels in loader:
            pass_ = model(inputs)
            loss = CE(pass_, labels)
            
            batch_size = inputs.size(0)
            total_loss += loss.item() * batch_size
            
            preds = pass_.argmax(dim=1)
            correct_preds += (preds == labels).sum().item()
            total_samples += batch_size

    if total_samples > 0:
    
            avg_loss = float(total_loss / total_samples) 
            accuracy = float(correct_preds / total_samples)        
    
    
    return avg_loss, accuracy


        


## Part 4 — Regularization (15 points)

### Q9 — Build a dropout-regularized MLP (8 pts)

Implement `build_regularized_mlp(dropout_probability)`.

Return an `nn.Sequential` model with the same dense architecture as Q2, but insert dropout **after each hidden-layer ReLU**.

The order must therefore be:

1. 54→128 dense layer;
2. ReLU;
3. dropout with the supplied probability;
4. 128→64 dense layer;
5. ReLU;
6. dropout with the supplied probability;
7. 64→7 output logits.

Do not add dropout after the output layer and do not add softmax.

In [ ]:
def build_regularized_mlp(dropout_probability):

    return nn.Sequential(nn.Linear(54,128), nn.ReLU(), nn.Dropout(dropout_probability), nn.Linear(128,64), nn.ReLU(), nn.Dropout(dropout_probability), nn.Linear(64,7))

### Q10 — Implement an early-stopping rule (7 pts)

Implement `early_stopping_epoch(validation_losses, patience)`.

Read the validation losses in chronological order. A new loss is an improvement only if it is **strictly lower** than every earlier validation loss.

- Reset the non-improvement counter whenever a new best loss appears.
- Stop when there have been `patience` consecutive epochs without a new best validation loss.
- Return the **1-based epoch number** at which training should stop.
- If the stopping condition never occurs, return the total number of supplied epochs.

Assume `validation_losses` is non-empty and `patience >= 1`.

In [35]:
def early_stopping_epoch(validation_losses, patience):
    # YOUR CODE HERE
    improvement = np.inf
    chances = 0
    for i, loss in enumerate(validation_losses):

        if loss < improvement:

            improvement = loss
            chances = 0

        else:

            chances += 1

        if chances == patience:

            return i + 1

    return len(validation_losses)


## Part 5 — End-to-End Training & Model Performance (20 points)

***This part puts the previous functions together*** to a complete deep-learning workflow: model creation, repeated mini-batch training, validation monitoring, model selection, and final test evaluation.

### Q11 — Train the final model and report its performance (20 pts)

Implement `train_final_model(train_loader, val_loader, test_loader)`.

Use the components from the earlier questions to train a regularized MLP end-to-end:

- set `torch.manual_seed(42)` immediately before creating the model;
- create the regularized MLP with **dropout probability 0.20**;
- use **Adam** with learning rate **0.001**;
- train for at most **20 epochs**;
- after every epoch, record training loss, validation loss, and validation accuracy;
- treat a validation loss as an improvement only when it is **strictly lower** than the best validation loss seen so far;
- keep a copy of the model parameters from the best validation-loss epoch;
- stop after **3 consecutive epochs** without validation-loss improvement;
- restore the best model parameters before the final evaluation;
- evaluate the restored model on both the validation and test sets.

Return exactly:

```python
model, history, report
```

where `history` is a dictionary with:

```python
{
    "train_loss": [...],
    "val_loss": [...],
    "val_accuracy": [...]
}
```

and `report` is a dictionary with these Python scalar values:

```python
{
    "epochs_trained": int,
    "best_epoch": int,
    "best_validation_loss": float,
    "best_validation_accuracy": float,
    "test_loss": float,
    "test_accuracy": float
}
```


In [36]:
import torch.optim as optim
def train_final_model(train_loader, val_loader, test_loader):
    # YOUR CODE HERE
    history = {}
    history["train_loss"] = []
    history["val_loss"] = []
    history["val_accuracy"] = []

    best_val_loss = float('inf')
    best_epoch = 0
    best_val_acc = 0.0
    best_model_state = None
    chances = 0
    epochs_trained = 0

    torch.manual_seed(42)
    model = build_regularized_mlp(0.2)
    optimizer = optim.Adam(model.parameters(), lr = 0.001)



    for epoch in range(1, 21):  
        epochs_trained = epoch

        
        train_loss = train_epoch(model, train_loader, optimizer)
        val_loss, val_acc = evaluate(model, val_loader)




        
        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(float(val_loss))
        history["val_accuracy"].append(float(val_acc))

        if val_loss < best_val_loss:
            best_val_loss = float(val_loss)
            best_epoch = epoch
            best_val_acc = float(val_acc)
            best_model_state = copy.deepcopy(model.state_dict())
            chances = 0
        else:
            chances += 1

        
        if chances >= 3:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

   
    # final_val_loss, final_val_acc = evaluate(model, val_loader)
    # realized I never needed this
    test_loss, test_acc = evaluate(model, test_loader)

    
    report = {
        "epochs_trained": int(epochs_trained),
        "best_epoch": int(best_epoch),
        "best_validation_loss": float(best_val_loss),
        "best_validation_accuracy": float(best_val_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc)
    }

    return model, history, report


        
    

In [37]:
# PROVIDED FINAL TRAINING / REPORT CELL — run after completing Q1-Q11.
# Do not modify this cell.

_FINAL_TRAIN_LOADER = DataLoader(
    TRAIN_DATASET,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_STATE),
    num_workers=0,
)

FINAL_MODEL, TRAINING_HISTORY, FINAL_REPORT = train_final_model(
    _FINAL_TRAIN_LOADER,
    VAL_LOADER,
    TEST_LOADER,
)

print("Final model performance")
print("-----------------------")
print(f"Epochs trained:           {FINAL_REPORT['epochs_trained']}")
print(f"Best epoch:               {FINAL_REPORT['best_epoch']}")
print(f"Best validation loss:     {FINAL_REPORT['best_validation_loss']:.4f}")
print(f"Best validation accuracy: {FINAL_REPORT['best_validation_accuracy']:.4f}")
print(f"Test loss:                {FINAL_REPORT['test_loss']:.4f}")
print(f"Test accuracy:            {FINAL_REPORT['test_accuracy']:.4f}")

Final model performance
-----------------------
Epochs trained:           20
Best epoch:               19
Best validation loss:     0.5210
Best validation accuracy: 0.7810
Test loss:                0.5418
Test accuracy:            0.7710


---
## Submission

1. Restart the runtime/kernel.
2. Run the provided setup and all required answer cells.
3. Make sure every required function is defined without errors.
4. Save the completed `.ipynb` file.
5. Submit the notebook to Gradescope.

**Before submitting:** run the provided final training/report cell after Q11 and save the notebook with the printed validation/test performance visible.
